# Qxern v6 — optional cross-receiver bridge appendix

This notebook reproduces a **negative compatibility control**: a lightweight bridge from the released Qwen3.5-0.8B packet space to Qwen3.5-2B.

The experiment is **not** a valid test of whether a larger receiver is better. Its interpretation is limited to whether a cheap linear/alignment bridge preserves the known sample-specific semantic control.

The public result showed that the bridge did not preserve that control. A meaningful receiver-scale comparison would require a receiver-native adapter or another alignment method that first passes the correct-packet-versus-other-packet semantic gate.

Run this notebook separately from the core audit in a fresh Colab GPU runtime.

## Procedure

Run all cells from top to bottom. The notebook pins public revisions, isolates the 0.8B and 2B phases in separate processes, and downloads a result ZIP with alignment, semantic-control, identifier, and generation diagnostics.

In [ ]:
# 1. Minimal pinned install. Run in a fresh Colab runtime.
import subprocess, sys
if 'transformers' in sys.modules:
    raise RuntimeError('Delete the runtime, reconnect, and run once from the top.')
subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-cache-dir','--upgrade-strategy','only-if-needed','transformers==5.14.0','accelerate>=1.10,<2','huggingface_hub>=0.34,<2','datasets>=4,<5','safetensors>=0.5,<1','pandas>=2.2,<3','scipy>=1.13,<2','psutil'])
print('Pinned runtime installed. No restart is required.')

In [ ]:
# 2. Experiment specification and isolated worker.
import base64, gzip, json
from pathlib import Path
WORK_DIR=Path('/content/qxern_receiver_scale_probe'); ARTIFACT_DIR=WORK_DIR/'artifacts'; LOG_DIR=WORK_DIR/'logs'; RESULT_DIR=WORK_DIR/'results'
for d in (WORK_DIR,ARTIFACT_DIR,LOG_DIR,RESULT_DIR): d.mkdir(parents=True,exist_ok=True)
SPEC_PATH=WORK_DIR/'spec.json'; WORKER_PATH=WORK_DIR/'worker.py'
SPEC_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/81ajXLbuBF+FVQ3N5bmZJn6l9VT2iSX62UmuUtjZ3Id28NAJCjxTBIMANpWc57pa/T1+iTdBUgKFCnbcZPpMRmZBBa7i/35sAD5qZVwxZacX7pXTMiQJ605afV7TqtLWpIxHx5HA7iPuc8il7ohtrT+fs2SI/wZ9MaHz6FLHPZ742cti1Cwq7Dg5wdDj02mnsMGo9HMCQbOaDJbTlgwmB2PJoOBPzlmy+lwuR2/dGVMo2hX3BDEOb3ZszqhLW4QeM5kOBlN++Pl8WQa9GcThzJ67AXD2XQ6m03H49ls2Z/aXCIqVqxJ3OBZncwWJpjk0RU7pOpQZIkKY4b0yyzxIyRMORLRmzAOjz7eMJEcXk0OfZZGfHMYJmpUId5yZVPHocHAD6bLSX827tNhfzQI0ErMmXpLNmNOfxhM9BSoT1PFhBuEEUsoyJ+XbdLIdAuS4UDxy16q7GFyTQfjCQ5azsbj8aDvsJnjD5yJN6RTNgFfDYe+c0z9CRuMHfDXdAJPnj+azo690WwQjMCgzvHUmyFXnyoqmcotueJ8FbEjD6zn3rirKGOup1z9qLir2I2yx3g8CcIVjks3ag2GsPoq/p1MZ87MYdMZHS0HDvWnx34/cOi4PwqckT/rj4/7MCVt25jeFPIuWSJhOPRAuxI0TNwgSzwFXIEEfAedw0HZl/DEY+4VFSFNlHRTtHBOjlxKOoUhoVwUVMoYzdCtIvQhWAwVBIK3xq7+tidiVCRhsnIFVeg1p+c4jtV/zcLVWrk+8+jGdFudK0H9kCVgtShMkW/PsWSiIrnkYmqz/vFgl+KKRmV/3xmMoN90B9RTXKC+Z4VW+Nf80b/wAyIvYIQxVMp55OaIMXAGE3DQcKcz/CfTndo6DGWb3lLDHduXmk1KRlKxGOyWrNQ6H5AKHqcKYilOIzCjVvlTy4Qfu4FpQCiCmcIgZAIDQgcd9P3EBCOhJJRgeBDIVMhkIKToYGiL8Vkyn0CWcoKsE0WumDbL/Dz5/vvvXz09ffHz6cmTJ0/Ok/drqpCbWjOipZI3OoRJETJkqwVhCYpEzpq+Kvov5C2gw4Zch2pNeAJ3SLMd3WvddkkxQcE8fmUFpqvz35rl07oecisQdFga/tX5kSWL+HV9lm+NOBIqmc+yZIuCe+SXTKWZKtQGk+jm1u2FXk1iSKTQc43HvpIPfM7QC8B0d+I+3zEtAwmJF0pGfCY9EaZI1kPrba2d59BXVfmBYXN/YORwl7DrLRLp3IH1XUmYSeqGgPhaVd0HiY2gAdAWZ2YGVYKx6ZcUcqvILJy2XtNZQLw18y7dFACSCx5FbSpEByZJ4Dpv6X/6/i1TmUjIqchA30CrDZR0Y+woFU18Knyi+RwiIxJTJcIbiCegFdfgoq7hJAynH2kkIdreUnTe6SZlL4TgouAdJjoGl7+B1VEEVDggJsnidGPk9s6TupIwGAlhkUlQI4/hbLokSXuoHYwqZqb1qEpun7c+AM0HEmdSQfIQHlgDiQK681anIqcN/aGU2dLHXpTV03da4hLhsrMVhxfM764hML0lE52KkpaxStk4KvHDmPxpQQYPIYbyIGVn/Qsc0B/M7hwito7Wazf3IOpggcN4+cMFAWpY1GeSRll/OByNxkfppoxnpMCFECP/U6sorXaCHolSKmis86MFvFt6UXTLRly011S6Rm9oUGADFI6am1Gl+q2LW0y4ENd8cVeakd+JFsCwygMZ8Gz4ozVhac3YnGwACn83sQokWxkgoVoS9QfHelGxcztLofRibcmioAu4FkXMrDCh3yUqVBFb/AwQ2rWxM29JRXgFQxfaQXZA2umG1zstgtAEsC+UCgLFEqTXoGSFjQCkeo3iqxXzoWommQS42/I5XetF7WMWCmaw9OBaAJC5W27ygEiPp6xMfbzm2oLVyZEzE7IXYLA1s/r+869/S/LyBwg4I8jv1Rhps+wwMG3cBOWWW4+0Sz6dGiPLpnfrYy9cpP2LvqFRA8fcJ+QMoYXRBNi9XzNMLLRuTC+ZWTQt++dDLL5/BnkBzSJY9rVrbQuY4INYfl5yyFV+l0iozOSabHt6+2MiExFZQOuR5bujb+V5i3xbddR2CG4RYMynKl6et7TpzzHf8Ka7223ZDomsxxppbgokK25LilsbECUaZ0EwaXouAFAbptPVCi7wp1MDz61NXuMOswfekqxtGHVqGPr/SBgbJjf0Mkth33gZhUdmi3aY5d5txkoDIjsQidYxGGS5Uxet6Cc96a0v9Fhj811cHd+Lq01gei+uVSCtRLNdwB3vAdwGcJ2OauDKhXsNuVeocck2XQLGhsqLQ82hudWB8zVmaTGSeBHNcN21cvCvJteBW72x4F7v0dLsZqwpGlWaExOKVaF5JONM9qc1rMqwa2ubxPiYMREy2SFPiDOv5poh8DIhcGuLhBs3THx2Q76D8mNLa+hgO8gFM2TtT+ctnDmYWOt+3iqnDE3bmZy3zIShUd/c1pOyCNBK7jVZv1JC9GVfbSAtDn+TPPnYnA/F+D0ZAYrjn0JZvNc67kb+6FGR/5Cg24ny0cOj3HFqUS6vado2FF1YY258wKzFoLpDeMNEwEWMJSBQ5/ogmsF/aAOMgt2yWUBzRuxmDbZX4RWDLRAOShHyKBSMhgIyvADAGM/dspjIlHm4R/IJqoDFfc9goKk9s6VkCkUgE2szJHt2GFPY1glVzIM8gfrZdORiFySC9ny+22K/pNcJkPdaUV/0L+x+0x01NQZY4+LRgaDJirWdbkQOSd/miBTWLJDWnlTO7iych9/lwi86O2kIetscoPRXGWwBHzIUL9iBlkbJh8zDi7voILstgfdQFloUOswbWG9CFvnbYRUkKk2+2OMTvJaC0csaCvyxw9XGo1TwjccTAKQkSvdsZVCdHSjKE9ocJaD4XewZ7GKPrgXvBZ8mLNhFm0ET2iS8AWzG/RrY5OfwsFp4+kwkR7ni0UW4q4dJedW3MT5TNIykfqovxuQ5LgAyP8ALAFbBTX4pDl2KB2yi0MvHLa5iVhF2whIfppn3E3aFwwCgzaEguG8Zsdiconkoy8/3taEKabRlA6EJVZ4g4BOoAGF0EN5UNicFhjUucz0PwhwKPy29XSl8zotXG+etuwy3NaA2V9Xi9RL2D242O4N8wdMlv4EMAkI/U5vmJNoNvD1ru22Whho3N9+XWekfkgvVKjcX//j1f1IvcnHTIXPRukSDYpqu2KLfRdRyzb1z1wHB3xge2kgIHdhDIz2iHrLNt1pSr3XUcG/a2usOawt9wqjw1gQmF8v6/l1LOAsh/sDhQP0Gn83BGgYmGALK1qvGHTFgUsOmO5+lzfNnww7mAfutWCKNlruHqdNp2mP7oaeA16fsQHFFo4M5cbokf9ASZdGU2wkez96B3S5uH7DzltpIR9p9NkG+zzaltyEy+1vbufhj+be46VTZnGlwQc3OWxfAE3WzNsCurqSQsLdiqm0Rd+rVOpLVYObzIud/2+OinfYkvWZv+lam6M/t8WXyfE+Gbe3/P9Tzk/oSi685Ya+gt2ZQDnLqg6PicAsvAYc5rLX37kvrLOXIAGsdgGjNpjglk1maRlj95LJkU2ZXhEFiG0+DT3XodHSC5SRM3nHeVAlqWD08JqVbYd6uzqt5GXVTDpQHHo9jmvhHYKnTXPl36Suc3IF9DNQYrp9lETtir/qX+txax+rHZ6HKqZrDdZ8T9y5b1uTvrQUfFLWfFUYPKREbA3g6qMUvcKVJwk21bKQ9h7yWXain95R5bwtZuI3SGA5/AX6pPny7zo9QZQaLCkWqgrs5oINqCKIBtxySd8maX0Plm1QXqhPGyBxmt55/eBpF/PoU9X9a8ultWcoPH3LQso+H8wjZ2qi3G+FFmGZxm/Q1Bxe3hDps4Yd5ql1aoYuvZ7r58uOGqwSPVywVOp0973W+sIU+dyNT8eyeSNaTNJ811QJ5+KhAbognYkwJ1q2F7vDhtdSwFrrgNv31AxWNaZIf3yzQfXch74/Ihmg2xPj+K8Hs3DpbnFfE6ibSxvcQna8Cy7psyE95Px1ocVD+4KFcz8/iVLbNSdftw5AcHKm1P0Hl7wbxu437eMS2fP8gkN53avi4IH9Y5D0+2GeTWrDrdxFtvvytS/CbtpSqddNnBe+RTKNf/poXanRKXr/84aUeVobzm1Iv83xYXvmJWi6D6PNt+y0J4iq0A9/rUlad/91fEehpvAY4DZWg3uXdXxGcwELPRf4BzQfrVXRvy+EDHj5LmU9alh8WwHPPWK40Wi1E7zPZY9+Ja7k7wQkSNHGuzCPOke55Ld4UJg87UXrYW/GGCgKLlTwLiq8t63D7Cqs4CqD3KnxKfn39qhoteD0VqyyGfN85cixYkjZ4rGPenmIbegkl74fLbyDuXv36Qq+lGsXwe1U8tsVNVe/XN09Pf7JkfaN/rrm4pIJnMASLgps4mod+eYDS9kNJ4c7v7I4LAJd5ypJ2aYFdCjz2DXqCga068Af2TpAHB7mEBYDoAcG/dc49L+KStXc79IyUgEppQV6AcmC5U3jKX5Y2qcEglOY7b4tyBqAF3gWCxziyabgm18yBuF0Oxr2o4ByWvU7lOPl+g2udMBJQbfM1yjVkoOQRxClp62xPoOqH9UKfZIFsPK2CVnQ7yx2ksah4d9bw2iyfIBaytcRvjsnPLbNQmz1LUPnx8Zc5Lm5KtMefFtfPphRP20mXXIe+Wi/ev/zh9Ceo2dQmYouT03+8erHzjgqsqMyZJYwjAp0XgEyFCWKnYl5HrEUGgVWw70KtlDCL+cmZfrjoAbM6RN8pzHZYEoosPtLNKY5pdhnw2PGYro+1ZroYRlW+TKVg2zS35uPLgpFze3H7X+TiQKueMAAA')))
WORKER_PATH.write_bytes(gzip.decompress(base64.b64decode('H4sIAMRScWoC/909/ZPbtpW/71/B5qZD0uYyKydNe9zQE1/iXGfOTdPUvZsbRcPhSpTELkXSJLWrzUb/+70vgABFSWs7Te/Ok6xEAnh4eHhfeHiALpZNtXGSZLnttk2WJE6+qaumc9KyrLq0y6uyvbhQ75pVnTZtpp/bTn1dzdW3ddqui/xGPf69rUr1vWrVtyYtF9VGP2mIXb7JLgijOu0QjELne3jUeNTttssL3aZq5mvrISxLwM0pdccd9Ncuq2aTNS2DX6Rd2mZdq+AXVbpI5CXXWG9Xq7xcLdN5lqy3GpH1Ep+SRXVfYpvA2VSLrEjycllxO7Mv1ejVtqu+rsplvgro+5+wzbdV83W6bdPizZ/47dvqNivzn7Lm4uJikS2dtkzrdl11XpHeZEXktF3jO5cvne+qMosuHPh3t3FioUZ4lzfdNi2STbapmgfPpwp1U837Kt/DU9a2XtWGq6yr84XnS7X0AQcDNR/pGf+51KsbOfQZ9O/bh7bLNkl6l+ZQVGTJ6gZq3W1C/cb51Jlcvfj82bPPjGY1d5408D+1wBchY0vk8/wQykbbzreLNEmLopqnXbbg5jzTWKKA6Aqe30Nx8qVZNW97xKFaVrQZ0XPYWZO1WXN3vC9V/sFd7WV+8rLzUEbCxXZTt57MhB84y2LbruO3zTbzFT+s0xe/+yJZ5gAPxYM4wvmZZIMYAx6ZLxb5Kms7mE4RxpCbClPc593aqeqsJCiB4zY3ro8CswapLIS18B9wsTNfb8tbJy+dvMsa4MTNzSKNpGbYZOnCw9E7z4gIgPaN6/o9hB6ZcFuDeGUewWM8mgxUTqnK19mOv3k43nmRAi+8gRZl95dvSZq8sgxBcLZFJh0gTRJgnbxLEq/NimUAsDbwgj+rbRc45XaTdChXLbzMahwulq3zxSKDestlqb/DNxhOG/8BqjRVDc3jq3BiDKbd1oCFH+o+fV0EM9+DdX6rYNmEkJcwLWW267w10XeNtPUmXwQO9Pt54LwInIl/AG/txLFz1feHow3fbbMmzwheGX6fNukmwzliFkQNW3rW8DVAH6brKrx6MYAHAvl3GBrDe5OXWdp4JkGlsd3oBmTuVnDg2XmT0xyaTJTgIAGjVebRJAw4xAAUpjUw5qKf62/yeec9WtVZSpuqbUE0seK26HIk7asO2QVMlmfOsZC9n1X5BF5Nu/ka5KlpOxa04LCfEnhvwv28SR+y5jt4NsCPNcHh/EqYvXg/zIDducFfM2AeQCgtPHuuDwUDZBqq/PvrN3/z+Os3jKQnyPJLAWHKk4HKMfQ/ezL6e3+MWaH7A241u0f07GbYq9HM6phqa80CfHufNgtRLOtkDjY7cNKuK5NN2t4aLMwmASCaUuRxi3AJ6rzzDOxRanVlEeFwW7bwNfsp8678MNvVILwKQJvT28C5nOD/PaA6XSzAQSFsAJ7GjFSFJX4kWiiChqDZEpgify4ycGcSAEU1piJiM48wDmSY/edt9pCYOMTmg2+BV2MWwCxTAth5rnv3z+NEsqVRMj/O9vjibI8jjT4zGslrFCJ5aUyrGDOLyTyLU3UTtubkcaaLtEad3dbZPHBIU6OdusvnyshJjQRNNRp02/30mqyuknwRI4Cpe7NFs5zgS3cGQgyeQglWQUoVKPUaqzTQVwt6aQiA32INQiqmvzzadE5+JrgUyMWGT2KiylXBjBm1fwPVLUS4sTvrebFJc/CSftiWuBB43TQVqJRPXnF1cEUyYNztxtnk7QYVZOQ89uD3CP9xvIP9J4xP1zwY5hyWNxkMge0lUdMcATB6WifkUSJ93Hm9dQPnPstX665NqrJ4EOcMYWW7Odg15+1DzVh/VC+adjAHJbQv55lHYFCpzTtw1coFWBl8k+ALl0Sb+sGSYaupWXUmMA4RHKlLdeZFlpYZrQ3YZ0WNArIfOHdpsc103yE4iJvWMyBTS2gHlS11VDfZMt+R3+NuyMiHQFk1afT9HS+gwqEvCUQhqCF02XQterIegxtUNPunz2mB/q5Undk4Zospfc6gMg3qwiSMVKF34/5UTz3R5+4MPe46E2ccFktQBT19qSY2ImRuUrWnk5mvW9yMtAAlMmxyJU1IN4EiQE9siu024aqptrU38X01Yz2bABm9jRPFIP4hSZLXuGwWfgy9HxfP/R9xFqCJ78/EINakfjbpztNd+aATJ8wS2uzbWItPdxVCBfh7BHeZeGhq+/tEOKbzzXt48n7YVR7rUKjSgUjGLHxkij97YXVKEpn0TK8EDZZS8GRIuKqfAYOId0u8jERVZbXywS0xqGGZ9G6bw4oxWTXQmfdtCktBawkkAHhE/ZB75SY2A2BUxV2WdLB+AO8AAwoef0h/sKxnSsIXsHGNlIJEwQuzGUwvrkW1qpnD4pkWqgzBtxeBt+AJrVqS2Ed3kaFf0oDrhtTZB6AXBsJnqVn1j7VFLCh6z54xUP+gImAjmqV1yqozwh3Df0K9XmTVv+PquHee2tacgQN6jdDKweUwvZHpYH5LyEOzpmG+XDnxqdm6GFCE+15ixwbQ4SRpstgkwbAV6ikCDczPjNwTVvWD9TDw41l90LC8vswcOJXB9I60OoPTiCF3v662xYKmVCgjBHQMiDIdqEMIsvKWVlmZNSikZQWGLamrqgDvCp3DeQWdoMBmmwR0/EqvLpuSJoEijeEP9EFNuCMYYVuVadmhxnRv5ovlav3322JT1u9g0XV3v3v4yeVZqu6zgiqlWV5t+SW6T6CHABxqvOkMvyqS36/BGXLQ3FAt3/mScTRNboaBO9cN/17BSgEQDefrCvSVZ2CF9h9W/i/InefoEaPC+jzvl9Tm0PtZx86xk3c/7RIXNDXWMuMVWB6Cr7DAdeAyx8gGuQ/UDieJ1gtZOVyqZ2UIfj6Nzb9mQuhFO74zp5FKVfQqK7J5xyo8IaOTLRKqYHq/MnfwvbWjkSnoGytIGmK0NQGL3gGvgX322PXjcGwK9mnEvVWFR/xbeGi2bQfFmwpYDdXccPXtgudZFO+Fy01CbU5hpKp8BF4FKNPsPfGiNqfx4iofhpeEOEFUScmNCDB11b9IUDyxj8MC1BCDAoPtXfEkyOVBpbAAlZqT1D6SgdL2mrgbOBuh9pzNftPjbeR0xMRgehfgFUBnOdhf9jpiNtthXtbbjjwtsoxAA4SHDKs84L0pZmn54KE03VH1O4qkLtqQlBvUHVhOkP8uLw1rBn4frq0AOw81CjSF1Q0MN3D0I3Ok9YqZQb0isewVgxApBI0Fcplui86TXgLQZP6hMLMDSASd4lt0kx+X7uPtXqQZ+sQdAEUQPUhFD+dnqz5WRrzu/KMtWMkCN9DCg2cdPS/hHtKn7gy1GhURe9tlvCwr8lV+Q+SbejX4kKI86Sv2qWgh/eKEEcWg3stYEJgpU4fTqCCetnTfVU63brLsslOSeCkazyEkuV9ABZbbYvgU4LCtms4D1zvmWDvG3TeR48G618Ov4DVPo89watEXt99cEgPgqwk+SqGvt4J4jueg5XOMxSPPq26hHptFsxBRAaXRv/OPG9ZxUcYVwu91K/D7t8slOJlDiGwbaK77omlk0r+Pr1DNaXSMKcgWS6VjdSKopOnxWEY9e89YS0iwjKDsSecpMzavNpuqFMav0xx8fT3JrMVFRdILEsQA/dFiC0PRCxhETET/Dtb+oP8pRGDBQacroVIPDC1XK6xqBN2sJp4NYthPH3om0glqL1yewLgwFi0wfeeS3JcBlsI3ogUpYtkNxxqi1bL1pA/SbtdkNEdq2uhOO6I9qVN5iRtq2hfhtrMzLKj57ve/n4wwHsH1x9TLTZMvwNTxxLJCGaoZqwoylalrRHVID+D0MfzTsS3QFKBTymq7WqsxM20i59GAtgdwjwxv7x6KDFUaFRfDgdKqV7UcOjWixil6IPwy7WZ+Pymq4Wzodhy2LM63NEkcnZ0Do2FP+OjMvCg3hCW3bvJN2jwky23J7N+Bhtb+Zo0cWPbSrxZlWAlWZTfV4sEl+zgIzIET5KVtF34rUL/JlgEmRoSv2odybrxVoRSZGegRdK7iGnjy0dGf9NvEehNUwfiBgrYN9fYduFpv+0SD4xuiVcFONhi87J6++REHqfsSjllLMZDhsI2Gjm5glxijkm5KwMdwZchYLrKQAcZ2h5FZZndu79WQy5jPE+rUoy6u9coCnlh9EkZDYmMgb4jqYBDfQX+nsc8Xx3DPF0cxN/FTQRus0LMduskUEzDnRMLTGZIEJ5jybPqqMPAl6shxHlb6B6qohbhom/9EN1Ockte7mjVGtkvnXfHgQD1kQ/Aa7rLCUSBd38RlyHzLMhywU8jTQ4hcE+7LfJds8rbFvSAV2W4NTFXQC5MCSh4oFzK9cPckLcByLGhFkWAOA1FCUQnogQvbH0tZPxewtAkxYFd7rHPwBVuRhS4I27rIOyxRfp7RysJL5UxsFr/zsK8wKxGQ52675eUfXN8fpCqwV1Dw6hZ1Fo5bUZMXt4I5OSIInryrx9GBgveOD66oz1aZPrBs6aYuMLa874HhTg/Bgma863OmFS10zGQnMZvyJItB+50Eg+g90DBm1e0erBlVfXO1CATO0g1QxAilagM0jJ8Y9OkjrOjF4nDMbAqkEEaa0cEVcqFmqCuKxDfkA3PMiohCYSx3W96W1b3ibyIGJUcxFPiOzFCuMDxtsowHjAas5oO2VgWmpqBeAT1jOtCEgFn32IAD3zovnckVBVfg40tS+NAd+ggvPr86twLEwH18QihMZFSQH8M2h5B07AYD+xdHg7WjWugJCsgIvL6mDyg+MzZLZyHdkDSiYog8Z9orTlJr1kdmhsjhzXpjUiOcbEx1Qn6IaNbgqWpg9VPi6gC3QyNHut5bREWcVEe0LDTXnIoOemVhIXwDzH97MQrnyyeCGd0R/XMJ6lt0Dij0Rwv03lFaSOv09hN/bBEFyusr3hrJy2UGdgpWSBj4AQ5HpcZ6DxY589usaz0KCRmLm6DftsB6La1IjS0a3oDhTBaM38Sfq+g4poC18XSmZZx28/qA5lXgKNlpfROEISv0MqYqU2oe0d/nfeV+k49CNm2sMfeoUiDkAANbwlKpjd26A6UmWQyxi6OR8BIFukrZo0VFxmPl0rgftrH11LNQte2g+5jI5z17xsgE8jpRQXXcdmoF9raFlWo6X2ccc9KILuwtKfzHzWMGFlrAppeTmUo/6TUn0V7Ji0o/UJtojNvUTVW6EuVygC0K5/UWFOI6LZZmLstB+uKjy0yzcCNQ+h5PCc4lUdwP+mn1A1iRdGnhRv27/SDt0XKXKa8SxsIjoD2y+Art71//+69vX/8p+f6HP//p+7egoj7572rrpE3mpM4cPpbbgu0F+NI5Ou1d6HwLolPdO906Q1o3biuzgcy4STvlIYWfXLx59fb1d2+T79+8+vr1H//85pvXP6Dr8eWXX3LBX1++fOn2ft4CVrvzdWosfjFdpm1TcBa0j2eo2wPdq0erArgwUcUDwQQ2BUuOKZQKYi+Lik8waimBVpy8uqk2dSdMBQ4cJut267y8Rf42diHPb5n9w/Cy0idGzIZ0rLZKlp98+XO+SYivfn75CHa7qQqw/vsfy0dyBUpkXHimajAhP7/8sfyEdAw5EXoyYE3vmqA0c4Cpl/nUsU1zNtlvtzJ0hrTxuM6ocnlCWNlUIWrhilkLSba5yUg1jaliVtIU1QLSJiNo4qt4nEenj0zGSHK73UBTMrKkax/omiQ3Rj2j4/1MRRxlnQNOEZfE5E+zW3UoWcILapisMSm8xcQxCADuCO/s63eSXBDSa9090DMem0eNmpDo2iEcj1ZXAwhM5X6gmqYaG08699XM2Bl+fYICj4L+Ql2jvaDjzwJUdBPFCzfbHJailJJv8yXGYbgFTzdXifv34bwAJwspBxYOU8Z6YcYXathyHmDBKX0CJeYmBuCpqjKLLydXV9qSN6BWtR3nqn22h7nhspOkjPs1OB6egjaF9jNfBaMNxwka+JHqGyoF8AKqzYxRZFVrj0Lmhxv1OeWUKvpvFDL6gJTyk/ng1xwUKKiL+CCLWmWl3+Rpa2i/g2xTHKqZT8jwvF2fTMrMgN3mtCpIeBu3e0g4FoY9DuSfC2Jr+FgL/j9wW+iEgBylqShrxUxoYUiClZI7mIGKKDBausjTVVVi5ky4zIsi8Sbh1UFdJIuCY04g15IxL3NDEajRcvyy1w/gP1Jc0nxjhBPBywD7Dlp5QKJdPASkKG4Q6NrBrRkb+kg1BtgFu7t4N42MzmfBbmo8RrByfeiCh7v4YVDtwa7GADcxKNBNlpYe0u9hEz8Yj7s5lF7uNlAwh4LLh41E2O9ibzcP3361m/ufojRRQInSe6XgoS9gJZyvYD3bBu9ESgtcJa1CnEzw1e5kHT9PiyzmoUsDxgXcxSLd1Am6gJPscvLCpw025Tm+6xLu+x30TN/EMwJexmUB8EXWdjEFQZVe4QlD1SJT17MjdBUTKs8YFa7Qezb38buvPNXnpwrTaRRgB7Pn0NzyaLMF0OQOqOh/df/8YWPEHFpg01gdMAvV2iotQi6CRccmB84AKUT1v4A5Vbpbr+VjJMnvJi/IEd7d+agxoMoYUL3iJ2DT6FZZgq9O1364M+qGb01PqskxSiEz5kHHIfAxbiNO/FhwSFlz34qhi4Wbfc3hMsHmfBL06j5+dJn0bmRNRODyTiZ4+ukmoPA9AkmYaqoyPw3Aq9qgPo7Wx7JhCz1WMAj1JEF1BoLUbm/aDFwVXbo3HHHkPbUsgsFYEQBkRzNKgRWmBwOZBfr1yf5n/ksPIeK+7QgQq+gMoEG0AaUGcQvuQ9zQnK9JEsnqi0LFshjrHTUJO52LGjz0aam/gIGYV/VD4t2Hb8f1Phc/bEDwvrofUf+B8+iquAVMIQzE5UlzI/4MJJQCRfSu1/ZMSXotYrc/E/lo51WDS3C1FT3mcJNTbuymJ5ITZlkUdCinYx5lqTzJsHcL+82nAVDW/SCogH5M2/6w6t6F5XaTgUXldjtJoWD/LsYa4w4eMvC40ySZxBRYkVQShgbreZVCuy0KT8InjI8fQIXATLMFhlsN1Me1Q94kl6OFb4dAzgBgz5PCLVbgKA9o3BmSArOLvN4BphaxTSk1mmkeRMUs3jFe8jS5Vj0oPVDoFcmZtQiSJTbWAcwc6iAPj/RyAv/5ATT0GInetZfTLqUJgr9WuF0hAASolslzNAvIPTf7qHQYigM9PIo2RvQDO+YTIz7BMBA1Hocqaow/AQqrvOtdIXhM2mrZIb/CwIH0yL2DQVyjShrMKGdg7YLCt6f2p7z2htIG0x3IpJmpVLB6su0Z9g0TgB/PC/gyZE/0COKiBk6ApsFuZtq11oguW927EWFqvFV77EXgtttNAiSAZfGNMlnkIGEej4TTwFiR6h+rpvwoqre39CGi9CT9hQvWE5pLloZmqMAM4RoazAhtGG3GAhzH4hvBMH56JHw6GgX5GBkkebPW0QNhIwnDtsKT0YvZk5TR/z5ZujjL+UhXVfuEEFyREGAiISywrWySX5SrA52SgkraxM1XKRt09CZJy/ZewmrClLxnUNV8hujH8i9bcGkwbugG8PQ1TAt+62OA6kGCfVQLA1g/lvT1bxzL+pJio1T5U/lq6hQOXi1z0ATYt+Uh1r+JLyeRo2Nc06i2sz0o7iW7h6dlVyfKyrBZgK3QFfI1Sg3uwct+i+V1GMaW2NtilPdm8FBhNOB0QWTA6/hngFw8xLVKeFdaJAFP9txk6aaNJ4HpqcSn3BQhrM0huj6fUfFgDMDDQXub10O1RCKnAin3sGjLEtzHKOROBFSyQk18EXIN0qh94oGx8dEEABQvUEnbeZ5LUJcjYhQPQ3DPsWVAGyS0wST5BEaUt0BP2T4MCgyb40IKFEXDG/N8vlLFq2jeeCb5waMLLKIrF9eW6gYRV0O/JBnoz2+cTveOY1fOalym3WXDu5CuEQI73bq/G+V45rl5vhRZ1DxtFfB5KPzLqZVQHh8enGVSmcw7X66SNO6vXXnKYYHzZwXM1Pcjme+69/apvVvHA556OuB9MCmejokxLU89D/BUTPLBcSmcIP83MR+OxHswBoWtFNLQT2db/wDLwrTNFvogIbQDaQT89Elhtz9YSSOIh90pV/9OkgHaQHLK6bvKFZY0YRx0G5870GIcmVaZtXF/Ssg8jND3+tzoNTL35qi5ukMFu9d5/7ByLu+yhrwmQEb0m2cl7EtlI11/5vtnwMsZgqeBZ7Y4BE8J0vF4znQ/hMDqMVD4BJQ6ddHrGSORmygEGqYn3fXIWQBVy6CqAY7rCFY8w1D3UUu+ck+kDC2JSUejZDILTBIYJWBs3fkafLE55l337z+TfKwDTPKfyEsGRITEalwkKfgaP82XQsReWsxCoWmsv+tek6FdM6wZ8S84OmDY4xdjlm3EiiFcz7RWn9p5uTTvIXbi+ke6pSrHYMvNSDA/mN6wBVq61S34aqt660bGtUu4PmBTQBOOV2e45p1cCXAz6a7IfBsm6n2SBMP7AyL4DGQCIjZJBuWj3jwZJI/0V8UaSmkmYlEXbnRGtx4yfGQoqBFOj0ylNcrf0QnWD9SEmfIqQTOamqny2Ge+ueotctD6iDTlF1QwgnmW3+EJYWKW/RhraH/kHEfQpGteHGGO6xOXaB3eoQX91mmTmX6DidgHuFUIznKqtBcav/d5xV/EBbl2RgJ/Y770L+JpEY7x2KV2v9aAg6K6T2BJiTffJVtM7+DXogQ2aQ06w42u9sHBZQCTLwK6NifH1Qj4DJ3cwtEu6hR40jjrb6fpoUk7nfCrTp7Ex8+oPP/Xf8W56iU2PmrADBQoU4o2yfh5A1bCjJ/dgCcEtA1U2mwfPxsMwTfTkDRkFesyU2t9s68+GHYLQokpI9kmBW9sDqpY+nYjhYTLWSFK66iqkgCEGocDSm50kIQbSKomF8iNNsOETS60X86M7M06n9+C10UngGjF6ZkK8pDcd2mT4+HrpMZrclRC/My+zEwfISXgw9zZA0oO0/9NygZ8NPMkffsD2k+gcF9Zjs8cUppjlh9JXAHOmaHxaKJoH7FQKsYgjtAeAwN8xkHsiqVaMj7Ko/g9472ycXbn6bUZfpAAP7jTQmfGBNoNH7Q8dDcHG249gscmmzs/mG49lH62D2dWIwhmXeOq3prxZ55L1dWR2VTFaOlgJOACmPNJCC2LtHvqXPZDf/pUanZRnYXgCWFMijLlh7MV9C9HJiLogej9yuGrFyK4mhLvya9TfRbj6KGK2ZNG37uultwod00elUOHfKGK8HswyBHX7fQLcRQ1WPNJyhio/mqwh241fHPCDR46vntxfNCwtuldpv22A6cPvaVFWCtfaSSBmMeW7YTE5H32xPCtsSY00VAHr5w0R80c4B8M02owHLA02o/5jTiqZJAdJEtZ2WGhBzteZ22hIG14r0Q50/2py+MRWY7AWlmHgw2zg9x5tXUbvLj6/A+GxjK3sk5tpciG/RQ3AMzMegQ3mxmZBQcp4QYD9BsxFPoMRheGB9lWxBeSfUtTLvldQwmVlADFTySEQrGA0v2MfQLaJODxWvcznbyXievLjTqDnRudxi21sHEOZjeh++nqKsdDKquE8629I6V8+RFoBATeZFABpq9TK6Yjl8o9qUNzAEREIylPnlkr1p3aI6i7fBO+WqSb//KkokmpoGhi65xtkaUN+bxoK4F3mV9g0T1PH+yaZolyo8CrgWlqYxJByRPlq9loLqemFpz5aC3PuNHeFSWJ4rTHsYSU+BjtBFsXVdtmlsSAgZyvjetzrDPErH2whm3vzVPkMgTbLWy7rMas1EE+glS1HQfyYsZHDGK3m11L1vBBHaWsudqxtEPdDUr7IhYh4rZ2Dd7DiY9lmPcSx6ACxHHqaqdyrFN2UIw9Y24j7udsdNt49MzNiS1j03MXh6jfRYY+3jPN3vLpPmajmXHxp1EQXU5ms2H6X0/xZDQXQuVsD7NAOOtmEvRt/QCzrs9mgBzJFn8Cmie3twVP/ji/sc0yKPvakuxpnbVW1/bn7RJTmkEhQLWRCx2P3KBQXnIz6sZJO+eRMNuTSDqP+HcfOY8sLAR67w5GW0sCM2Wy4RnWrkKHMzMHEN6AAFGSNoVYJAcTL9bH3Pq8ZitCd7+OqlFLz/Q6HFrSEhtxQFQ9GzVWX/06gVQTeHL4EbjYAPwyVD2yXmRpo+8YlKZLsvuRD6y3v/eHs4G3idXPJ/5vJ1/E8VU05qERX2A2J/4ugIHC84l1pkupPskHwKggjJcxwnyB3ZTRm/X5azzY6eXki2jmfzr5YuCKPc3wLvLWMIQHR4v6q/8YCgd5tIlUj+rwAGEkPgmWbTHi+QFuSZeuLCcId8H7I1HC7GYY6HDRc815PZIiDctEOs1OTz0kflaRFm331NKnNyjWkkGbkmu9WjLrHiwW7IOFRjboMKXBuCWbLEhCGciWRTJVsfDltaqNHavaGrHD+tTJvzggnJttzeN9sk0jjKZXwZVay+mTTHysDY0tpuNiaLo3edfHckSNRI2Aw0ymleHrbwk+hrR3AT+MRR8O0RgEMD4s9MEpntAShTI2SSDwZuo8gd3kKQTVcGVYQ6JZEI3TQmdJeTOgZdSzf68ZQcIwUrJSMTGKnChEKNp8GGR59gxQ2B8i1gdcjHiLTWK+3HHkZtH3c6ks2ge639lTSPi+ZBRPZUBKViofR8enBKuMh4+LWY3OmvoVDlOr0pULprihRRu9STYFvfnExKs+4NOnNJl6wVDFvzJNGW83gs/90Ok8DLMMMBpDR+MysOxD7TJipv9FmyCwpGISHEkJxOsZGnAHOm1rxDw5uAW7bYGf8QD7wqm6ddZI2UdGfAlU7Onx+L8dG4j9yzcVCBLOYyDYQVdTzxXc3cAwUVp3+oHnUld2Mb2a+YPbIZ6sKeRw77FNlDHVgOdxDrKAD3lZBMzcdxF/5cwSyx/cCivuxhjHHzK2pqwb9UQeVwmmqJtZelY8aek+Qk/7RDQZ1XADfgKJPNtO260BBG1kngLElHtpb7x6CgQ9s4NxKOKq0Bisizl09nG72Mq9i82fJjgarw1Gfg3B+jEEXmBfS8oP6RoE2XpPTgrBX5NidtXxxNv4A+/Z/QUT6Wg/3d5JH+yhzwu0Huq+uD6pJHD/cp+VnyW/M3bFkfK8Zw6t/hnj+Yjt8s9ePGG73EyPIPQukQ0ydc/p2UA6x9EtdpYoOp9wjk+cej5IitKnrBRuR5dwgYI1vnxzFaU51dC113K93ZMjdR+C47U6zx0/3kZ3wzi7nOOTG3PpwlxZsRo/XuDbF+hyotQgY2p/eA2JhEjHQ+4nV7USbh1fauNPysE7+6dGbHyWaV6AkVwCEIyuYNhFiOAGboYRngSZECwJ/MULSvG0PW2xJlKOoYeGSvYH5w8HP+0gkI1wuF5Hav6wo8X9pgallsFSDOaRYl3eaMKbZAeqnBAe5ukMJ65zKsFpuLUyMu3BGBoqBtW5p4zPEGeKEonlYWL0O7jjgnOeP5JuWx6Rmd6iUfbb/2OLBnoRaXEctxObYsGxHwl6Xwv5j0nw/udZyH/IeH5NC0no2Rby2HZxf9Xu7Px52/fdJf4Vdoc58X6wMWwYzQAvCYmP3+8RcDatykS1bgfmOWcBkgsiTMM6Lmlgn1clTtJpDY1onU5AfS/lzNMoKKBJ6rMhPlDHCmMAqIGKNTa0f0HX4vrX8Cs4ifqX9Cv0bAul/g+6F3LE4J/tXlgcTKb9rJMxwPwXdjIE+hknY0O5D+pMX9NmTax+0Dx81ay2yBvf03u+JAu/4XWpSSqFntvhbYgB/1YPzGKfxe4GyrVwlU8c6F/hOALs8hIH4QaShbJQtvRI5fuqub2EMR02oEwSaUUf2K71/IuDDdaR36WOxn+q6W/fvHK2pa7n9sdwTEcIf3naw85CSno2/J0DNiM8+5mJ+6Y4MPLxPnVVjda1a4ebW/jrIX3LTm7JzHZ47AD8HzPpPa7aMCvv8gZQpPt///ht8vbP//H6u/7AI/WJExnHxvxFw/OXhz5oMBit/t2OA5jEB9HZswbj7dUPDJ2P8kj7Noue4ECDEOR4txlrN+gnSVAgkgS6Ysm4+B9GPYxTC4AAAA==')))
print('Worker and embedded specification written.')

In [ ]:
# 3. Worker launcher.
import os, psutil, subprocess, sys, time

def memory_snapshot(label):
    vm=psutil.virtual_memory()
    payload={'label':label,'available_gb':round(vm.available/1024**3,3),'used_percent':vm.percent}
    print(payload,flush=True)
    return payload

def run_worker(task):
    log_path=LOG_DIR/f'{task}.log'
    cmd=[sys.executable,str(WORKER_PATH),task,'--spec',str(SPEC_PATH),'--work-dir',str(WORK_DIR)]
    env=os.environ.copy()
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
    memory_snapshot(f'before-{task}')
    with log_path.open('w',encoding='utf-8') as handle:
        p=subprocess.Popen(cmd,stdout=handle,stderr=subprocess.STDOUT,text=True,env=env)
        while p.poll() is None:
            time.sleep(15)
            print(f'{task}: still running',flush=True)
    memory_snapshot(f'after-{task}')
    tail=log_path.read_text(encoding='utf-8',errors='replace').splitlines()[-50:]
    print(f'--- {task}.log tail ---')
    print('\n'.join(tail))
    if p.returncode!=0:
        raise RuntimeError(f'{task} failed with exit code {p.returncode}. Inspect {log_path}.')
    return log_path


In [ ]:
# 4. Resolve revisions, verify dimensions/checksum, and select three-tokenizer-matched names.
run_worker('preflight')
preflight=json.loads((ARTIFACT_DIR/'preflight.json').read_text())
print(json.dumps(preflight,indent=2))

In [ ]:
# 5. Freeze the released sender and encode training/evaluation packets.
run_worker('prepare')
assert (ARTIFACT_DIR/'prepared.pt').exists()
print('Prepared packets complete.')

In [ ]:
# 6. Qwen3.5-0.8B: released baseline, matched bridge tuning, and evaluation.
run_worker('small')
assert (ARTIFACT_DIR/'small_bridge.pt').exists()
print('0.8B arm complete.')

In [ ]:
# 7. Qwen3.5-2B: embedding-aligned initialization, matched bridge tuning, and evaluation.
run_worker('large')
assert (ARTIFACT_DIR/'large_bridge_tuned.pt').exists()
print('2B arm complete.')

In [ ]:
# 7. Analyze matched readout scaling and export an auto-downloaded ZIP.

import hashlib, importlib.metadata, json, shutil
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

spec=json.loads(SPEC_PATH.read_text(encoding='utf-8'))
names=spec['eval_nonce_names']; chance=1/len(names)
TAGS=['small_released','small_tuned','large_init','large_tuned']

def read_jsonl(path):
    if not path.exists() or not path.read_text().strip(): return pd.DataFrame()
    return pd.DataFrame([json.loads(x) for x in path.read_text().splitlines() if x.strip()])

def normalize_answer(text):
    v=str(text).strip().replace('`','')
    if v.startswith('def '): v=v[4:]
    v=v.split('(')[0].strip(); v=v.split()[0] if v.split() else ''
    return v.rstrip('.,;:')

def matrix_for(frame,prompt_id,base_id,score_kind):
    sub=frame[(frame.prompt_id==prompt_id)&(frame.base_id==base_id)]; m=np.empty((len(names),len(names)))
    for target in range(len(names)):
        g=sub[sub.target_id==target]; d=dict(zip(g.candidate_name,g[score_kind])); m[target]=[d[n] for n in names]
    return m

def loo_center(m):
    out=np.empty_like(m)
    for row in range(len(m)): out[row]=m[row]-np.delete(m,row,axis=0).mean(0)
    return out

def grouped_bootstrap(frame,value_col,iters,seed):
    groups=np.array(sorted(frame.base_id.unique())); means=frame.groupby('base_id')[value_col].mean().to_dict(); rng=np.random.default_rng(seed); boot=[]
    for _ in range(iters):
        s=rng.choice(groups,size=len(groups),replace=True); boot.append(np.mean([means[int(g)] for g in s]))
    return float(frame[value_col].mean()),float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))

def permutation_for(frame,prompt_id,score_kind,iters,seed):
    mats={int(b):matrix_for(frame,prompt_id,b,score_kind) for b in sorted(frame.base_id.unique())}
    obs=[]
    for m in mats.values():
        for row in range(len(names)): obs.append(m[row,row]-np.delete(m[:,row],row).mean())
    observed=float(np.mean(obs)); rng=np.random.default_rng(seed); null=np.empty(iters)
    for i in range(iters):
        vals=[]
        for m in mats.values():
            assignment=rng.permutation(len(names))
            for row,col in enumerate(assignment): vals.append(m[row,col]-np.delete(m[:,col],row).mean())
        null[i]=np.mean(vals)
    return observed,float((1+np.sum(null>=observed))/(iters+1)),float(null.mean())

all_scores=[]; all_gens=[]; all_sem=[]
for tag in TAGS:
    s=read_jsonl(ARTIFACT_DIR/f'{tag}_scores.jsonl'); g=read_jsonl(ARTIFACT_DIR/f'{tag}_generations.jsonl'); sem=read_jsonl(ARTIFACT_DIR/f'{tag}_semantic_scores.jsonl')
    if len(s): all_scores.append(s)
    if len(g):
        g['normalized_answer']=g.answer.map(normalize_answer); g['exact']=(g.normalized_answer==g.target_name).astype(int); all_gens.append(g)
    if len(sem): all_sem.append(sem)
scores=pd.concat(all_scores,ignore_index=True); generations=pd.concat(all_gens,ignore_index=True) if all_gens else pd.DataFrame(); semantics=pd.concat(all_sem,ignore_index=True)

rows=[]; per=[]
for tag in TAGS:
  sf=scores[scores.tag==tag]
  for prompt_i,prompt_id in enumerate([p['id'] for p in spec['prompt_templates']]):
    for score_i,kind in enumerate(['mean_logprob','sum_logprob']):
      lifts=[]; raw=[]; loo=[]
      for base in sorted(sf.base_id.unique()):
        m=matrix_for(sf,prompt_id,base,kind); c=loo_center(m)
        for target in range(len(names)):
          lift=float(m[target,target]-np.delete(m[:,target],target).mean()); lifts.append({'base_id':base,'value':lift}); raw.append({'base_id':base,'value':int(np.argmax(m[target])==target)}); loo.append({'base_id':base,'value':int(np.argmax(c[target])==target)})
          per.append({'tag':tag,'prompt_id':prompt_id,'score_kind':kind,'base_id':base,'target_id':target,'target_name':names[target],'self_score_lift':lift,'raw_correct':raw[-1]['value'],'loo_correct':loo[-1]['value']})
      for metric,data,null in [('self_score_lift',lifts,0),('raw_top1',raw,chance),('loo_top1',loo,chance)]:
        df=pd.DataFrame(data); val,lo,hi=grouped_bootstrap(df,'value',spec['bootstrap_iterations'],spec['seed']+len(rows)+1000); p=np.nan
        if metric=='self_score_lift': _,p,_=permutation_for(sf,prompt_id,kind,spec['permutation_iterations'],spec['seed']+prompt_i*100+score_i)
        rows.append({'tag':tag,'prompt_id':prompt_id,'score_kind':kind,'metric':metric,'value':val,'ci_low':lo,'ci_high':hi,'null':null,'p_value':p})
per_example=pd.DataFrame(per); summary=pd.DataFrame(rows)

# Open-vocabulary exact generation.
if len(generations):
  for tag,g in generations.groupby('tag'):
    val,lo,hi=grouped_bootstrap(g,'exact',spec['bootstrap_iterations'],spec['seed']+9000+len(rows)); rows.append({'tag':tag,'prompt_id':g.prompt_id.iloc[0],'score_kind':'greedy','metric':'open_vocab_exact','value':val,'ci_low':lo,'ci_high':hi,'null':np.nan,'p_value':np.nan})
summary=pd.DataFrame(rows)

# Semantic reference log-prob lift: correct minus rotated-other packet.
sem_wide=semantics.pivot_table(index=['tag','base_id'],columns='condition',values='mean_logprob').reset_index(); sem_wide['semantic_lift']=sem_wide['correct']-sem_wide['other']
sem_rows=[]
for tag,g in sem_wide.groupby('tag'):
    val,lo,hi=grouped_bootstrap(g,'semantic_lift',spec['bootstrap_iterations'],spec['seed']+12000+len(sem_rows)); sem_rows.append({'tag':tag,'semantic_reference_logprob_lift':val,'ci_low':lo,'ci_high':hi})
semantic_summary=pd.DataFrame(sem_rows)

# Primary paired scale comparison on per-base mean lift: large_tuned - small_tuned.
base_lifts=per_example[per_example.score_kind=='mean_logprob'].groupby(['tag','prompt_id','base_id'],as_index=False).self_score_lift.mean()
scale_rows=[]
for prompt_id in [p['id'] for p in spec['prompt_templates']]:
    w=base_lifts[base_lifts.prompt_id==prompt_id].pivot(index='base_id',columns='tag',values='self_score_lift').dropna(subset=['small_tuned','large_tuned'])
    d=w['large_tuned']-w['small_tuned']; rng=np.random.default_rng(spec['seed']+15000+len(scale_rows)); boot=[]
    for _ in range(spec['bootstrap_iterations']): boot.append(float(rng.choice(d.to_numpy(),size=len(d),replace=True).mean()))
    # exact sign-flip test over 12 base-level deltas when feasible
    vals=d.to_numpy(); null=[]
    for mask in range(1<<len(vals)):
        signs=np.array([1 if (mask>>i)&1 else -1 for i in range(len(vals))]); null.append(float((vals*signs).mean()))
    observed=float(vals.mean()); p=(1+sum(abs(x)>=abs(observed) for x in null))/(len(null)+1)
    scale_rows.append({'prompt_id':prompt_id,'large_minus_small_tuned_lift':observed,'ci_low':float(np.percentile(boot,2.5)),'ci_high':float(np.percentile(boot,97.5)),'sign_flip_p_two_sided':p,'n_bases':len(vals)})
scale_comparison=pd.DataFrame(scale_rows)

bridge_alignment=json.loads((ARTIFACT_DIR/'bridge_alignment.json').read_text())
print('Primary receiver-scale comparison'); display(scale_comparison)
print('All candidate-free summaries'); display(summary.sort_values(['score_kind','prompt_id','metric','tag']))
print('Semantic control'); display(semantic_summary)
print('Embedding bridge alignment'); display(pd.DataFrame(bridge_alignment['trials']))
if len(generations): print('Generation examples'); display(generations.head(24))

# Plots
plot=summary[(summary.metric=='self_score_lift')&(summary.score_kind=='mean_logprob')].copy()
fig,ax=plt.subplots(figsize=(11,5)); labels=[f'{r.tag}\n{r.prompt_id}' for r in plot.itertuples()]; err=np.vstack([plot.value-plot.ci_low,plot.ci_high-plot.value]); ax.bar(range(len(plot)),plot.value,yerr=err,capsize=4); ax.axhline(0,linestyle='--'); ax.set_xticks(range(len(plot)),labels,rotation=25,ha='right'); ax.set_ylabel('Mean-token self-score lift'); ax.set_title('Qxern receiver-scale probe'); ax.grid(axis='y',alpha=.3); plt.tight_layout(); plt.savefig(RESULT_DIR/'receiver_scale_self_score_lift.png',dpi=160); plt.show()
plot2=summary[(summary.metric.isin(['raw_top1','loo_top1']))&(summary.score_kind=='mean_logprob')].copy(); fig,ax=plt.subplots(figsize=(12,5)); labels=[f'{r.tag}\n{r.prompt_id}\n{r.metric}' for r in plot2.itertuples()]; ax.bar(range(len(plot2)),plot2.value); ax.axhline(chance,linestyle='--',label='chance'); ax.set_xticks(range(len(plot2)),labels,rotation=35,ha='right'); ax.set_ylabel('Top-1 accuracy'); ax.legend(); ax.grid(axis='y',alpha=.3); plt.tight_layout(); plt.savefig(RESULT_DIR/'receiver_scale_accuracy.png',dpi=160); plt.show()

# Export
scores.to_csv(RESULT_DIR/'all_candidate_scores.csv',index=False); generations.to_csv(RESULT_DIR/'all_generations.csv',index=False); semantics.to_csv(RESULT_DIR/'all_semantic_scores.csv',index=False); per_example.to_csv(RESULT_DIR/'receiver_scale_per_example.csv',index=False); summary.to_csv(RESULT_DIR/'receiver_scale_summary.csv',index=False); semantic_summary.to_csv(RESULT_DIR/'semantic_control_summary.csv',index=False); scale_comparison.to_csv(RESULT_DIR/'primary_scale_comparison.csv',index=False); pd.DataFrame(bridge_alignment['trials']).to_csv(RESULT_DIR/'bridge_alignment_trials.csv',index=False)

for name in ['preflight.json','bridge_alignment.json']:
    shutil.copy2(ARTIFACT_DIR/name,RESULT_DIR/name)
for name in ['small_training_loss.jsonl','large_training_loss.jsonl','small_training_status.json','large_training_status.json']:
    if (ARTIFACT_DIR/name).exists(): shutil.copy2(ARTIFACT_DIR/name,RESULT_DIR/name)
for log in LOG_DIR.glob('*.log'): shutil.copy2(log,RESULT_DIR/log.name)
(RESULT_DIR/'samples.json').write_text(json.dumps(spec['samples'],indent=2,ensure_ascii=False),encoding='utf-8')
training_status={name:json.loads((ARTIFACT_DIR/name).read_text()) for name in ['small_training_status.json','large_training_status.json'] if (ARTIFACT_DIR/name).exists()}
metadata={'notebook_version':spec['notebook_version'],'created_utc':pd.Timestamp.utcnow().isoformat(),'spec_without_samples':{k:v for k,v in spec.items() if k!='samples'},'bridge_alignment':bridge_alignment,'training_status':training_status,'primary_comparison':'large_tuned minus small_tuned candidate-free mean-token self-score lift, paired by base function','interpretation_boundary':['The released Qxern sender and 32-token packet generator are frozen.','Only a linear receiver-side bridge is trained for each receiver.','Training identifiers and evaluation identifiers are disjoint.','The 0.8B control and 2B receiver use the same training functions, prompts, targets, epoch count, and optimizer settings.','The 2B bridge is initialized by shared-token embedding alignment; bridge alignment quality must be inspected before interpreting a null result.','A scale benefit supports receiver-side capacity as one bottleneck, but does not prove model size alone is sufficient.','A null result does not establish architectural impossibility because only one bridge family and small training set are tested.']}
(RESULT_DIR/'metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False,default=str),encoding='utf-8')

# Generate concise machine-readable interpretation.
sc=scale_comparison.set_index('prompt_id'); both_positive=bool((sc.large_minus_small_tuned_lift>0).all()); both_ci=bool((sc.ci_low>0).all())
large_open=summary[(summary.tag=='large_tuned')&(summary.metric=='open_vocab_exact')]; large_open_value=float(large_open.value.iloc[0]) if len(large_open) else float('nan')
bridge_cos=float(bridge_alignment['selected']['val_mean_cosine'])
training_ok=all(v.get('status')=='ok' for v in training_status.values()) and len(training_status)==2
interpretation={'bridge_val_mean_cosine':bridge_cos,'training_status':training_status,'matched_training_completed':training_ok,'large_minus_small_positive_both_prompts':both_positive,'large_minus_small_ci_above_zero_both_prompts':both_ci,'large_tuned_open_vocab_exact':large_open_value,'suggested_reading':(('Receiver scale improved candidate-free readout under the matched bridge protocol.' if both_ci else 'No robust receiver-scale improvement was established under this matched bridge protocol.') if training_ok else 'At least one matched-training arm failed and fell back to its initialization; treat the scale comparison as incomplete.'),'caveat':('Embedding alignment is strong enough for a directional scale interpretation.' if bridge_cos>=0.75 else 'Embedding alignment is weak; a null large-model result is inconclusive.')}
(RESULT_DIR/'interpretation.json').write_text(json.dumps(interpretation,indent=2),encoding='utf-8')

(RESULT_DIR/'README.md').write_text('# Qxern 0.8B vs 2B matched receiver-readout probe\n\nStart with `interpretation.json`, `primary_scale_comparison.csv`, `receiver_scale_summary.csv`, and `bridge_alignment.json`.\n\nThe released Qxern sender and adapter are frozen. A linear receiver-side bridge is trained on the same 32 functions and the same mixed semantic/identifier objective for both Qwen3.5-0.8B and Qwen3.5-2B. Training nonce names are disjoint from the eight evaluation names. The primary endpoint is the paired difference in candidate-free identifier self-score lift between `large_tuned` and `small_tuned`.\n\n`small_released` reproduces the original packet without readout tuning. `small_tuned` is the matched 0.8B control. `large_init` uses only shared-token embedding alignment. `large_tuned` adds the matched readout training.\n',encoding='utf-8')

zip_path=Path(shutil.make_archive(str(WORK_DIR/'qxern_qwen35_08b_vs_2b_receiver_scale_results'),'zip',root_dir=RESULT_DIR))
def sha(path):
    h=hashlib.sha256();
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
print('Results ZIP:',zip_path); print('SHA-256:',sha(zip_path))
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print('Download manually from:',zip_path)
